# Chapter 4 – Worked examples

This notebook runs the worked example of [Chapter 4](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-1-math/ch04-numerical-linear-algebra/) live: the Sherman–Morrison update of the
inverse of a resistor network's reduced conductance matrix after removing one branch, the zero denominator that
marks a bridge, the Woodbury version for two branches, and the float32 round-off near a bridge. The code lives in
the modules next to it (`sherman_morrison.py`, `rank_one_lab.py`, `woodbury.py`, `near_bridge.py`,
`sparse_fill_in.py`, `floating_point.py`); the cells only call them.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q matplotlib networkx numpy pandas scipy tzdata

In [ ]:
import floating_point as fp
import near_bridge as nb
import numpy as np
import pandas as pd
import rank_one_lab as rl
import sparse_fill_in as sf
import woodbury as wb
from sherman_morrison import bridge_indicator, fr, g, n, reduced_incidence, remove_branch, to

## The worked example: remove branch (1, 3) with a rank-1 update

$G_r' = G_r - g_k a_k a_k^\top$, so $(G_r')^{-1} = G_r^{-1} + \dfrac{g_k\, G_r^{-1} a_k a_k^\top G_r^{-1}}{1 - g_k\,
a_k^\top G_r^{-1} a_k}$. The update agrees with a fresh inverse to round-off.

In [ ]:
ar = reduced_incidence(fr, to, n)
gr = ar.T @ np.diag(g) @ ar
minv = np.linalg.inv(gr)
k = 3  # branch (1, 3)
keep = np.arange(len(g)) != k
fresh = np.linalg.inv(ar[keep].T @ np.diag(g[keep]) @ ar[keep])
print('max |Sherman-Morrison - fresh inverse| =', np.max(np.abs(remove_branch(minv, ar, g, k) - fresh)))

## The denominator $1 - g_k R_{\text{eff},k}$ per branch

$a_k^\top G_r^{-1} a_k$ is the effective resistance between the ends of branch $k$. Branch 6, `(4, 5)`, is the
only path to node 5, so $g_k R_{\text{eff},k} = 1$ and the denominator vanishes: removing it islands node 5.

In [ ]:
print(rl.denominator_table(ar, g, minv, fr, to).round(6).to_string(index=False))
print('bridge indicator of (4, 5):', bridge_indicator(minv, ar, g, 6))

## Three ways to remove a branch

Rebuild and solve, update the inverse, or update the solution vector directly; all three agree.

In [ ]:
ways = rl.remove_three_ways(3)
print(pd.DataFrame(ways, index=[f'v{i}' for i in range(1, 6)]).round(6))

## Two branches at once: Woodbury with a closed-form 2×2 solve

The inner $2 \times 2$ matrix is $C^{-1} - U^\top G_r^{-1} U$. Its determinant is zero when the two removals
island part of the network, which the flag reports instead of raising, as ToOp's `solve2x2` does.

In [ ]:
for j, kk in ((2, 5), (0, 1), (3, 4)):
    updated, ok = wb.remove_two_branches(minv, ar, g, j, kk)
    print(f'remove ({fr[j]}, {to[j]}) and ({fr[kk]}, {to[kk]}): valid = {ok}')

## Near-bridge: float32 against float64

With a weak branch `(5, 0)` of conductance $g_w$, the denominator for removing `(4, 5)` is about $g_w / 2$;
float32 round-off grows as it shrinks, float64 stays below $10^{-9}$.

In [ ]:
print(nb.precision_table().to_string(index=False, float_format=lambda x: f'{x:.4g}'))

## Factor once, update many times

On a 40 × 40 grid, 100 single-branch removals: rebuilding and re-solving against one sparse LU plus the
rank-1 solution update. The second method needs one pair of triangular solves per removal and no inverse.

In [ ]:
table = rl.time_removals(*rl.grid_network())
total_update = table.attrs['factor_s'] + table['update_s'].sum()
print(f'rebuild: {table["rebuild_s"].sum():.3f} s   factor once + update: {total_update:.3f} s')
fig = rl.plot_timings(table)

## Fill-in and floating point

In [ ]:
print(sf.fill_in_table().to_string(index=False, float_format=lambda x: f'{x:.2f}'))
for expr, value in fp.surprises().items():
    print(f'{expr:45s} -> {value!r}')